# 00 - Environment setup

Run this once per machine or pod. It checks the GPU, verifies the environment,
runs the test suite, and pre-downloads the pretrained models so later notebooks
never stall mid-run.

## Creating the environment (shell, once)

```bash
conda create -n adaptts python=3.11 -y
conda activate adaptts

# CUDA build of PyTorch. cu121 works on Turing (GTX 16xx) through Ada (4090).
pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu121

pip install -r requirements.txt
python -m ipykernel install --user --name adaptts --display-name "AdapTTS (conda)"
```

Then pick the **AdapTTS (conda)** kernel in Jupyter before running anything
below. The first cell prints which interpreter you are actually on, so a
wrong-kernel mistake shows up immediately rather than as a confusing import
error later.

Expected: about 5 minutes plus roughly 4 GB of model downloads.

In [1]:
!nvidia-smi

Wed Sep 23 02:18:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1660 Ti   WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   58C    P0             24W /   80W |     516MiB /   6144MiB |      2%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import sys, os

print("interpreter:", sys.executable)
print("python     :", sys.version.split()[0])
in_conda = "adaptts" in sys.executable.lower() or os.environ.get("CONDA_DEFAULT_ENV") == "adaptts"
print("env        :", os.environ.get("CONDA_DEFAULT_ENV", "(none)"))
if not in_conda:
    print()
    print("WARNING: this does not look like the adaptts environment.")
    print("Select the 'AdapTTS (conda)' kernel from the kernel picker.")

interpreter: c:\Users\as\.conda\envs\adaptts\python.exe
python     : 3.11.16
env        : adaptts


In [3]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
os.environ["PYTHONIOENCODING"] = "utf-8"
print("repo:", REPO)

repo: g:\Adaptive-TTS


## Verify dependencies

If anything is missing here, run the pip commands from the shell block above in
a terminal, then restart the kernel.

In [4]:
import importlib

required = [
    ("torch", "torch"), ("torchaudio", "torchaudio"), ("transformers", "transformers"),
    ("numpy", "numpy"), ("scipy", "scipy"), ("soundfile", "soundfile"),
    ("pyarrow", "pyarrow"), ("yaml", "pyyaml"), ("tqdm", "tqdm"),
    ("tensorboard", "tensorboard"), ("huggingface_hub", "huggingface_hub"),
]
missing = []
for mod, pkg in required:
    try:
        m = importlib.import_module(mod)
        print(f"  ok      {pkg:<18} {getattr(m, '__version__', '')}")
    except ImportError:
        missing.append(pkg)
        print(f"  MISSING {pkg}")
if missing:
    raise SystemExit("install these first: pip install " + " ".join(missing))

  ok      torch              2.5.1+cu121
  ok      torchaudio         2.5.1+cu121
  ok      transformers       5.17.0
  ok      numpy              2.4.6
  ok      scipy              1.17.1
  ok      soundfile          0.14.0
  ok      pyarrow            25.0.1
  ok      pyyaml             6.0.3
  ok      tqdm               4.70.1
  ok      tensorboard        2.21.0
  ok      huggingface_hub    1.32.0


In [5]:
import torch

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print("No GPU visible. Training will run, but very slowly.")
else:
    p = torch.cuda.get_device_properties(0)
    cc = p.major + p.minor / 10
    print(f"gpu     : {p.name}")
    print(f"memory  : {p.total_memory / 1024 ** 3:.1f} GB")
    print(f"compute : {p.major}.{p.minor}")
    print()
    # torch reports is_bf16_supported() True on Turing, but that is emulation,
    # not hardware. Only Ampere and newer have bf16 tensor cores.
    if cc >= 8.0:
        print("Ampere or newer: use train.precision = bf16 (no gradient scaler needed).")
        print("  -> configs/exp1_egyptian.yaml is already set up this way.")
    else:
        print("Turing or older: NO hardware bf16, despite what torch reports.")
        print("Use train.precision = fp16 with a gradient scaler.")
        print("  -> configs/exp0_small.yaml is already set up this way.")
    if p.total_memory / 1024 ** 3 < 10:
        print()
        print("Under 10 GB: start with configs/exp0_small.yaml.")
    x = torch.randn(1024, 1024, device="cuda")
    torch.cuda.synchronize()
    print()
    print("GPU matmul check:", bool(torch.isfinite((x @ x).sum())))

torch: 2.5.1+cu121 | cuda available: True
gpu     : NVIDIA GeForce GTX 1660 Ti
memory  : 6.0 GB
compute : 7.5

Turing or older: NO hardware bf16, despite what torch reports.
Use train.precision = fp16 with a gradient scaler.
  -> configs/exp0_small.yaml is already set up this way.

Under 10 GB: start with configs/exp0_small.yaml.

GPU matmul check: True


## Run the test suite

These check causality, KV-cache equivalence, collation alignment and, most
importantly, that the model actually learns homograph disambiguation on a
controlled corpus. All must pass before you spend GPU time.

In [6]:
import subprocess, sys, os

tests = [
    "tests/test_egyptian.py",     # text normalization, the waw rule
    "tests/test_models.py",       # causality, KV cache, masking
    "tests/test_data.py",         # collation, bucketing, config
    "tests/test_end_to_end.py",   # does it actually learn to disambiguate
    "tests/test_integration.py",  # the real pipeline on synthetic data
]
env = dict(os.environ, PYTHONIOENCODING="utf-8")
for t in tests:
    print()
    print("=" * 60)
    print(t)
    print("=" * 60)
    r = subprocess.run([sys.executable, t], capture_output=True, text=True, env=env)
    print(r.stdout[-2500:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-2000:])
        raise SystemExit(t + " FAILED - fix this before continuing")
print()
print("All tests passed. The code is ready to train.")


tests/test_egyptian.py
PASS  test_abbreviations_respect_word_boundaries
PASS  test_basic_cardinals
PASS  test_ctc_targets_drop_unmappable_characters
PASS  test_dates_and_times
PASS  test_diacritics_are_stripped_and_arabic_digits_converted
PASS  test_elongation_is_collapsed
PASS  test_email_uses_dot_and_reads_names_as_names
PASS  test_empty_and_whitespace_input
PASS  test_every_number_under_10000_is_speakable
PASS  test_fractions_use_the_plural_denominator
PASS  test_negative_and_decimal
PASS  test_no_waw_between_magnitude_groups
PASS  test_normalization_is_idempotent
PASS  test_ordinals_and_digit_strings
PASS  test_output_never_contains_digits_or_latin
PASS  test_percent_and_currency
PASS  test_phone_groups_can_be_read_as_numbers
PASS  test_phone_numbers_are_grouped_not_spelled_out
PASS  test_romanization_emits_only_aligner_vocabulary
PASS  test_romanization_index_map_is_consistent
PASS  test_thousands_separators_are_not_read_as_decimals
PASS  test_time_says_alsaaa_only_once
PASS  tes

## Check the Egyptian text normalizer

This runs its own suite. The headline rule: no linking waw between magnitude
groups, so 2024 is "الفين اربعة و عشرين", never "الفين و اربعة و عشرين".

In [7]:
!python src/adaptts/text/egyptian.py

EGYPTIAN ARABIC NUMBER READING
Rule: the linking waw appears ONLY between a unit and a ten.
      Magnitude groups are juxtaposed with no waw.

  ok           0  ->  صفر
  ok           1  ->  واحد
  ok           2  ->  اتنين
  ok           7  ->  سبعة
  ok          10  ->  عشرة
  ok          11  ->  حداشر
  ok          15  ->  خمستاشر
  ok          19  ->  تسعتاشر
  ok          20  ->  عشرين
  ok          21  ->  واحد و عشرين
  ok          24  ->  اربعة و عشرين
  ok          76  ->  ستة و سبعين
  ok          99  ->  تسعة و تسعين
  ok         100  ->  مية
  ok         200  ->  متين
  ok         300  ->  تلتمية
  ok         800  ->  تمنمية
  ok         876  ->  تمنمية ستة و سبعين
  ok         101  ->  مية واحد
  ok         999  ->  تسعمية تسعة و تسعين
  ok        1000  ->  الف
  ok        2000  ->  الفين
  ok        3000  ->  تلاتة الاف
  ok       10000  ->  عشرة الاف
  ok       11000  ->  حداشر الف
  ok        2024  ->  الفين اربعة و عشرين
  ok        1986  ->  الف تسعمية ستة و تمانين
 

## Pre-download the pretrained models

In [8]:
import yaml
from transformers import (
    AutoFeatureExtractor, AutoModel, AutoModelForCTC, AutoProcessor, AutoTokenizer, MimiModel,
)

cfg = yaml.safe_load(open("configs/base.yaml", encoding="utf-8"))

print("1/4 CTC aligner")
AutoProcessor.from_pretrained(cfg["align"]["model_id"])
AutoModelForCTC.from_pretrained(cfg["align"]["model_id"])

print("2/4 SSL span encoder")
AutoFeatureExtractor.from_pretrained(cfg["spanemb"]["model_id"])
AutoModel.from_pretrained(cfg["spanemb"]["model_id"])

print("3/4 MARBERTv2 teacher")
AutoTokenizer.from_pretrained(cfg["teacher"]["model_id"])
AutoModel.from_pretrained(cfg["teacher"]["model_id"])

print("4/4 Mimi codec")
MimiModel.from_pretrained(cfg["codec_model_id"])

print()
print("all models cached")

1/4 CTC aligner


Loading weights:   0%|          | 0/423 [00:00<?, ?it/s]

2/4 SSL span encoder


Loading weights:   0%|          | 0/421 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: MahmoudAshraf/mms-300m-1130-forced-aligner
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 
lm_head.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02-twitter
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


4/4 Mimi codec


Loading weights:   0%|          | 0/350 [00:00<?, ?it/s]


all models cached


Setup is done. Continue to **01_prepare_data.ipynb**.